In [ ]:
import cv2
import numpy as np
import os
import shutil

BASE = '/kaggle/input/datasets/ggrill/foodseg103/FoodSeg103/Images'
OUT = '/kaggle/working/dataset'

# Create output folders
for split in ['train', 'val']:
    os.makedirs(f'{OUT}/images/{split}', exist_ok=True)
    os.makedirs(f'{OUT}/labels/{split}', exist_ok=True)

def convert_mask_to_yolo(mask_path, label_path, img_w, img_h):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    lines = []
    
    unique_classes = np.unique(mask)
    unique_classes = unique_classes[unique_classes != 0]  # skip background
    
    for class_id in unique_classes:
        binary = (mask == class_id).astype(np.uint8)
        contours, _ = cv2.findContours(
            binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        for contour in contours:
            if cv2.contourArea(contour) < 100:  # skip tiny noise
                continue
            points = contour.reshape(-1, 2).astype(float)
            points[:, 0] /= img_w   # normalize x to 0-1
            points[:, 1] /= img_h   # normalize y to 0-1
            coords = ' '.join([f'{x:.6f} {y:.6f}' for x, y in points])
            lines.append(f'{class_id - 1} {coords}')  # -1 = convert to 0-indexed
    
    with open(label_path, 'w') as f:
        f.write('\n'.join(lines))

# Convert training set
print("Converting training set...")
train_imgs = os.listdir(f'{BASE}/img_dir/train')
for i, img_file in enumerate(train_imgs):
    stem = img_file.replace('.jpg', '')
    img_path = f'{BASE}/img_dir/train/{img_file}'
    mask_path = f'{BASE}/ann_dir/train/{stem}.png'
    if not os.path.exists(mask_path):
        continue
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    shutil.copy(img_path, f'{OUT}/images/train/{img_file}')
    convert_mask_to_yolo(mask_path, f'{OUT}/labels/train/{stem}.txt', w, h)
    if (i + 1) % 500 == 0:
        print(f'  {i+1}/{len(train_imgs)} done')

# Convert test set (used as validation)
print("Converting val set...")
test_imgs = os.listdir(f'{BASE}/img_dir/test')
for i, img_file in enumerate(test_imgs):
    stem = img_file.replace('.jpg', '')
    img_path = f'{BASE}/img_dir/test/{img_file}'
    mask_path = f'{BASE}/ann_dir/test/{stem}.png'
    if not os.path.exists(mask_path):
        continue
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    shutil.copy(img_path, f'{OUT}/images/val/{img_file}')
    convert_mask_to_yolo(mask_path, f'{OUT}/labels/val/{stem}.txt', w, h)
    if (i + 1) % 500 == 0:
        print(f'  {i+1}/{len(test_imgs)} done')

print("All done!")

In [3]:
category_path = '/kaggle/input/datasets/ggrill/foodseg103/FoodSeg103/category_id.txt'

names = []
with open(category_path, 'r') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            names.append(parts[1])

names = names[1:]  # remove background class

yaml_content = f"""path: /kaggle/working/dataset
train: images/train
val: images/val

nc: {len(names)}
names: {names}
"""

with open('/kaggle/working/dataset/data.yaml', 'w') as f:
    f.write(yaml_content)

print(f"data.yaml created with {len(names)} classes")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/ggrill/foodseg103/FoodSeg103/category_id.txt'

In [ ]:
!pip install ultralytics -q

from ultralytics import YOLO

model = YOLO('yolov8n-seg.pt')
print("Model loaded successfully")

In [ ]:
results = model.train(
    data='/kaggle/working/dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device='0',
    patience=10,
    project='/kaggle/working/runs',
    name='platecalc'
)

In [ ]:
import glob
import shutil

pt_files = glob.glob('/kaggle/working/runs/**/best.pt', recursive=True)
if pt_files:
    shutil.copy(pt_files[0], '/kaggle/working/best.pt')
    print("Saved from:", pt_files[0])
else:
    print("No weights found!")